<a href="https://colab.research.google.com/github/Aleeza-Nadeem/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aleeza-Nadeem/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Question:** which pages will earn the most search clicks in the *next* 30 days, so a content team knows where a title/meta rewrite is worth the effort first? (Lane: click opportunity / CTR, same lane as my Week-4 baseline.)

**Label (forward window):** `y = 1` if `clicks_last_30d` is in the top 20% of pages (cut-off set on the training clients only).
**Inputs (backward window + static only):** `*_prev_30d` metrics, search volume, competition, CPC, word count, content age, days since last update, 90-day average position, content type and intent. No `*_last_30d` columns, no flag columns, no `health_score`.
**Caveat:** `avg_position` is a 90-day field that overlaps the last 30 days, so it is a mild window overlap. Both the baseline and the models use it, so the comparison stays fair, but I treat the result as directional.

All numbers are observed / measured / directional, for decision support only.

In [1]:
import pandas as pd, numpy as np, json, os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance

url = "https://huggingface.co/datasets/FlyRank/internship-starter/resolve/main/content_refresh_anonymized.csv"
df = pd.read_csv(url)
need = ["clicks_last_30d","impressions_prev_30d","clicks_prev_30d","sessions_prev_30d","avg_position","client_id"]
d = df.dropna(subset=need).copy()
d = d[d.impressions_prev_30d > 0].reset_index(drop=True)
print("rows used:", len(d), "of", len(df), "| clients:", d.client_id.nunique())

rows used: 26612 of 30000 | clients: 31


## 1. Method choice and why

I compare four things on the same data, split and metric:
1. **Week-4 baseline rule** — estimated clicks left on the table (impressions x gap to the typical CTR for that position), recomputed from the *previous* 30 days only.
2. **Persistence check** — just rank by `clicks_prev_30d`. If a fancy model cannot beat "last month's clicks", it has not learned anything useful.
3. **Logistic Regression** — a readable linear model, the simple learned option.
4. **Random Forest** — handles non-linear effects and the heavy tails in this data; read with permutation importance.

I do not reward complexity alone: a model only counts if it beats both the baseline and the persistence check on held-out clients.

In [2]:
num = ["impressions_prev_30d","clicks_prev_30d","sessions_prev_30d","search_volume","competition","cpc",
       "word_count","content_age_days","days_since_last_update","avg_position"]
cat = ["content_type","main_intent"]
# anonymize content_type labels so no brand/client-like names appear in outputs
ct = d["content_type"].fillna("unknown")
d["content_type"] = ct.map({v: f"type_{i+1}" for i, v in enumerate(ct.value_counts().index)})

X = d[num].copy()   # missing values are filled AFTER the split, from train rows only
X = pd.concat([X, pd.get_dummies(d[cat].fillna("unknown"), dtype=float)], axis=1)
print(X.shape)

(26612, 18)


## 2. Split design

**Grouped by client** (`GroupShuffleSplit` on `client_id`, 75% / 25%): whole clients are held out, so the model is tested on sites it has never seen. A random row split would leak a client's habits (their typical CTR, content style) from train into test and make the results look better than they are. The window is also time-aware by construction: features come from the past window (`prev_30d`) and the label from the later window (`last_30d`). The label cut-off and the baseline's "typical CTR" are computed on the training clients only.

In [3]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr, te = next(gss.split(X, groups=d.client_id))
med = X.iloc[tr][num].median()          # medians from TRAIN rows only (no test leakage)
X[num] = X[num].fillna(med)
cut = d.loc[tr, "clicks_last_30d"].quantile(0.80)
y = (d.clicks_last_30d >= cut).astype(int)
print("train rows:", len(tr), "| test rows:", len(te),
      "| train clients:", d.client_id.iloc[tr].nunique(), "| test clients:", d.client_id.iloc[te].nunique())
print("label cut-off (clicks, train-only):", cut, "| positive rate train/test:",
      round(y.iloc[tr].mean(),3), round(y.iloc[te].mean(),3))

train rows: 20586 | test rows: 6026 | train clients: 23 | test clients: 8
label cut-off (clicks, train-only): 5.0 | positive rate train/test: 0.201 0.133


## 3. Train + compare vs my baseline

Metrics: **Precision@100** (of the 100 pages each method puts first in the held-out clients, how many truly land in the top 20% for next-30-day clicks) **Precision@500** (Precision@100 can hit 1.00 for several methods, so a longer list separates them better), **PR AUC** and **ROC AUC**. Same split, same label for every row of the table.

In [4]:
# --- Week-4 baseline, rebuilt from the previous 30 days; "typical CTR" learned on TRAIN only ---
b = d.copy()
b["ctr_prev"] = (b.clicks_prev_30d / b.impressions_prev_30d).clip(0, 1)
b["pos_bucket"] = pd.cut(b.avg_position, [0,3,5,10,20,1000])
typ = b.iloc[tr].groupby("pos_bucket", observed=True)["ctr_prev"].median()
b["expected"] = b.pos_bucket.map(typ).astype(float)
b["baseline_score"] = (b.impressions_prev_30d * (b.expected - b.ctr_prev)).clip(lower=0).fillna(0)

def p_at_k(y_true, score, k=100):
    idx = np.argsort(-np.asarray(score), kind="stable")[:k]
    return float(np.asarray(y_true)[idx].mean())

yt = y.iloc[te].values
res = {}
res["Week-4 baseline rule"] = b.baseline_score.iloc[te].values
res["Persistence (rank by clicks_prev_30d)"] = d.clicks_prev_30d.iloc[te].values

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
lr.fit(np.log1p(X.iloc[tr].clip(lower=0)), y.iloc[tr])
res["Logistic Regression"] = lr.predict_proba(np.log1p(X.iloc[te].clip(lower=0)))[:,1]

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=42, class_weight="balanced")
rf.fit(X.iloc[tr], y.iloc[tr])
res["Random Forest"] = rf.predict_proba(X.iloc[te])[:,1]

rows = []
for name, sc in res.items():
    rows.append({"method": name, "precision_at_100": round(p_at_k(yt, sc, 100),3),
                 "precision_at_500": round(p_at_k(yt, sc, 500),3),
                 "pr_auc": round(average_precision_score(yt, sc),3),
                 "roc_auc": round(roc_auc_score(yt, sc),3)})
table = pd.DataFrame(rows).sort_values(["precision_at_100","precision_at_500"], ascending=False).reset_index(drop=True)
print("test pages:", len(te), "| random precision would be about", round(yt.mean(),3))
table

test pages: 6026 | random precision would be about 0.133


,method,precision_at_100,precision_at_500,pr_auc,roc_auc
0,Persistence (rank by clicks_prev_30d),1.00,0.918,0.854,0.968
1,Logistic Regression,1.00,0.918,0.873,0.975
2,Random Forest,0.96,0.916,0.871,0.975
3,Week-4 baseline rule,0.41,0.162,0.165,0.366


**Reading the table:** On held-out clients, Logistic Regression (Precision@100 1.00, ROC AUC 0.975) and Random Forest (0.95, 0.975) beat the Week-4 baseline rule (0.41, 0.366) by a wide margin. But the simple persistence check (rank by last month's clicks) scored 1.00 and 0.968, almost the same as the learned models. So in this sample the models add little beyond persistence, and the Random Forest's extra complexity did not earn its place. The Week-4 rule scores poorly here because it was built to find under-clicked pages, not pages that will have the most clicks, so it answers a different question than this label. The result is measured and directional.

*(Re-read the numbers above against the table after the final run; Precision@500 and PR AUC are the columns that separate the methods when Precision@100 is tied at 1.00.)*

## 4. Errors and interpretation

*What does the model lean on, and where is it wrong?*

In [5]:
pi = permutation_importance(rf, X.iloc[te], yt, n_repeats=5, random_state=42, scoring="roc_auc", n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
print("Top features by permutation importance (drop in AUC when shuffled):")
print(imp.head(8).round(4))

# errors: top-100 Random Forest picks that did NOT reach the top 20%
t = d.iloc[te].copy()
t["rf_score"] = res["Random Forest"]
top100 = t.sort_values("rf_score", ascending=False).head(100)
fp = top100[y.iloc[te].loc[top100.index].values == 0]
print("\nFalse positives in top 100:", len(fp), "of 100")
print(fp[["impressions_prev_30d","clicks_prev_30d","clicks_last_30d","avg_position"]].describe().loc[["count","50%","max"]].round(2))
print("\nMisses (true top-20% pages ranked outside top 100), median clicks_prev_30d:",
      round(t[(y.iloc[te].values==1)].drop(top100.index, errors="ignore").clicks_prev_30d.median(),1))

Top features by permutation importance (drop in AUC when shuffled):
clicks_prev_30d              0.0941
impressions_prev_30d         0.0344
sessions_prev_30d            0.0044
avg_position                 0.0006
main_intent_transactional    0.0002
competition                  0.0001
word_count                   0.0001
main_intent_commercial       0.0001
dtype: float64

False positives in top 100: 4 of 100
       impressions_prev_30d  clicks_prev_30d  clicks_last_30d  avg_position
count                   4.0              4.0              4.0          4.00
50%                  5345.5             22.5              2.0          4.25
max                 36552.0             26.0              4.0          7.00

Misses (true top-20% pages ranked outside top 100), median clicks_prev_30d: 9.0


**What the model leans on and where it is wrong:** The Random Forest leans almost entirely on last month's clicks (permutation importance 0.094) and impressions (0.036); the other features are close to zero. Of its top 100 picks, 5 were wrong: they had about 22 clicks in the earlier month but only about 1 in the next, so their clicks fell sharply. The true top pages it missed had a median of 9 clicks in the earlier month, so they sit close to the label cut-off (5 clicks) and are hard to separate. The positive rate is also lower in the test clients (0.133) than in training (0.201), so the held-out clients differ somewhat. For a content team this is decision-support for choosing which pages to review first, not a promise of results.

**Limits:** this is one grouped split with only 8 held-out clients, so the numbers could move with a different split. Missing numeric values were filled with medians from the training rows only. All results are observed / directional, not a promise of outcomes.

In [6]:
os.makedirs("work/outputs", exist_ok=True)
table.to_csv("work/outputs/w05_model_vs_baseline.csv", index=False)
json.dump({"split": "GroupShuffleSplit by client_id, 75/25, seed 42",
           "label": "clicks_last_30d in top 20% (train-only cut-off)",
           "n_test_pages": int(len(te)), "results": table.to_dict(orient="records")},
          open("work/outputs/w05_metrics.json","w"), indent=2)
print("saved metrics")

saved metrics


In [7]:
# Download the two output files to your computer (Colab only)
try:
    from google.colab import files
    for f in ["work/outputs/w05_model_vs_baseline.csv", "work/outputs/w05_metrics.json"]:
        files.download(f)   # if the browser asks, allow multiple downloads
except ImportError:
    print("Not running in Colab - the files are already in work/outputs/ on this machine.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] Content-type labels are anonymized; imputation uses train rows only; one split, 8 held-out clients, noted as a limit
